# phase 3 brrrt — blind judging of 10,560 rollouts by Gemma 4 31B
Two passes per rollout (temp 0 and temp 0.7), phase 2's rubric, one item per request.
Inputs uploaded to `/content`: `job_gemma_judge_vllm.py`, `gen.jsonl`, `hf_token`.
**Never calls `userdata.get`** — the token comes from the uploaded file, so the secrets dialog that deadlocked the previous run cannot appear.

In [ ]:
# === CELL 1 — install vLLM, keep the torch/torchvision CUDA builds consistent =========================
import subprocess, sys, time, importlib
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.29.0"], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-1500:])
import torch
cu = torch.version.cuda.replace(".", "")[:3]
chk = subprocess.run(["python3", "-c", "import torch, torchvision, torchaudio, transformers, vllm; print('ok')"], capture_output=True, text=True)
if "ok" not in chk.stdout:
    print("mismatch -> reinstalling matched vision/audio for cu" + cu, chk.stderr[-400:])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch=={torch.__version__.split('+')[0]}", "torchvision", "torchaudio",
                    "--index-url", f"https://download.pytorch.org/whl/cu{cu}"], check=False)
    chk = subprocess.run(["python3", "-c", "import torch, torchvision, torchaudio, transformers, vllm; print('ok', torch.__version__)"], capture_output=True, text=True)
print("imports:", chk.stdout.strip()[-200:], chk.stderr[-400:])
print(f"install {time.time()-t0:.0f}s")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


In [ ]:
# === CELL 2 — launch the judge (background subprocess; frees the GPU when it exits) ==================
import subprocess, os
tok = open("/content/hf_token").read().strip()
env = dict(os.environ, OUT_DIR="/content/judge", GEN="/content/gen.jsonl",
           MODEL="google/gemma-4-31B-it", MAX_NUM_SEQS="64", TP="1",
           HF_TOKEN=tok, HUGGING_FACE_HUB_TOKEN=tok, VLLM_ENABLE_V1_MULTIPROCESSING="0")
with open("/content/judge.log", "w") as f:
    p = subprocess.Popen(["python3", "-u", "/content/job_gemma_judge_vllm.py"], stdout=f, stderr=subprocess.STDOUT, env=env)
print("judge pid", p.pid)


In [ ]:
# === CELL 3 — poll =================================================================================
import subprocess, os
print(subprocess.run(["grep", "^\\[", "/content/judge.log"], capture_output=True, text=True).stdout[-3000:])
print(subprocess.run(["tail", "-n", "3", "/content/judge.log"], capture_output=True, text=True).stdout[-800:])
print("outputs:", sorted(os.listdir("/content/judge")) if os.path.exists("/content/judge") else "none yet")
